# Backend Testing Notebook
Run from the project root: `jupyter lab notebooks/testing.ipynb`

In [1]:
import sys
sys.path.insert(0, '..')
from logger import setup_logging
setup_logging()
print('OK')

OK


## Run full pipeline

In [2]:
from pipeline.runner import run_pipeline
ctx = run_pipeline('../data/raw/Data Structure.xlsx')
print(ctx['vars'])

2026-03-04 18:20:55 | INFO     | pipeline.runner | --> Stage: ingest
2026-03-04 18:20:55 | INFO     | pipeline.stage_01_ingest | Ingesting: Data Structure.xlsx
2026-03-04 18:20:57 | INFO     | pipeline.stage_01_ingest | Loaded  Inventory:112  Sales:10000  Purchase:378  Expenses:12
2026-03-04 18:20:57 | INFO     | pipeline.runner | <-- Stage: ingest (1.506s)
2026-03-04 18:20:57 | INFO     | pipeline.runner | --> Stage: clean
2026-03-04 18:20:57 | INFO     | pipeline.stage_02_clean | Cleaned sales: removed 0 fully-empty rows
2026-03-04 18:20:57 | INFO     | pipeline.stage_02_clean | Cleaned purch: removed 0 fully-empty rows
2026-03-04 18:20:57 | INFO     | pipeline.runner | <-- Stage: clean (0.103s)
2026-03-04 18:20:57 | INFO     | pipeline.runner | --> Stage: transform
2026-03-04 18:20:57 | INFO     | pipeline.stage_03_transform | Transform done. GrossRevenue=61,258,985  NetRevenue=59,510,472
2026-03-04 18:20:57 | INFO     | pipeline.runner | <-- Stage: transform (0.004s)
2026-03-04 18:

{'TotalQuantitySold': 21141, 'TotalQuantityBought': 12251, 'GrossRevenue': 61258985, 'TotalDiscount': 1748513, 'TotalRevenue': 59510472, 'TotalPurchaseCost': 24885688, 'WeightedAvgCost': 2031.318912741817, 'COGS': 42944113.13427475, 'TotalOperatingExpense': 4364012}


## Load into state and test each analysis module

In [3]:
import state
state.store.update(ctx)
state.store['ready'] = True

In [4]:
from analysis.profitability import compute_profitability
compute_profitability(state.store)

2026-03-04 18:20:57 | INFO     | analysis.profitability | Profitability: NetProfit=12,202,347  GPM=27.84%


{'gross_revenue_npr': 61258985,
 'total_discount_npr': 1748513,
 'net_revenue_npr': 59510472,
 'cogs_npr': 42944113.13,
 'gross_profit_npr': 16566358.87,
 'gross_profit_margin_pct': 27.84,
 'total_opex_npr': 4364012,
 'net_profit_npr': 12202346.87,
 'net_profit_margin_pct': 20.5}

In [5]:
from analysis.discounts import compute_discounts
compute_discounts(state.store)

2026-03-04 18:20:57 | INFO     | analysis.discounts | Discounts: 2.85% of gross revenue


{'total_discount_npr': 1748513,
 'discount_pct_of_revenue': 2.85,
 'discounted_transactions': 2501,
 'total_transactions': 10000,
 'discounted_txn_pct': 25.01,
 'avg_discount_per_txn_npr': 699.13,
 'monthly_discount': [{'MonthNum': 1, 'monthly_discount_npr': 72871},
  {'MonthNum': 2, 'monthly_discount_npr': 80658},
  {'MonthNum': 3, 'monthly_discount_npr': 106188},
  {'MonthNum': 4, 'monthly_discount_npr': 137931},
  {'MonthNum': 5, 'monthly_discount_npr': 57967},
  {'MonthNum': 6, 'monthly_discount_npr': 105128},
  {'MonthNum': 7, 'monthly_discount_npr': 74692},
  {'MonthNum': 8, 'monthly_discount_npr': 89533},
  {'MonthNum': 9, 'monthly_discount_npr': 123283},
  {'MonthNum': 10, 'monthly_discount_npr': 230231},
  {'MonthNum': 11, 'monthly_discount_npr': 270885},
  {'MonthNum': 12, 'monthly_discount_npr': 399146}]}

In [6]:
from analysis.inventory import compute_inventory
compute_inventory(state.store)

2026-03-04 18:20:57 | INFO     | analysis.inventory | Inventory: turnover=1.73x  DIO=211.5d


{'inventory_turnover': 1.73,
 'days_inventory_outstanding': 211.5,
 'items_below_reorder_level': 106,
 'below_reorder_items': [{'ItemID': 'ELEC002',
   'ItemName': 'Realme Narzo 70 Pro',
   'ClosingStock': 0,
   'ReorderLevel': 8},
  {'ItemID': 'ELEC004',
   'ItemName': 'JBL Tune 520BT Wireless Headphones',
   'ClosingStock': 1,
   'ReorderLevel': 20},
  {'ItemID': 'ELEC005',
   'ItemName': 'boAt Rockerz 255 Earphones',
   'ClosingStock': -87,
   'ReorderLevel': 30},
  {'ItemID': 'ELEC006',
   'ItemName': 'Portronics Harmonics Z7 Speaker',
   'ClosingStock': -20,
   'ReorderLevel': 15},
  {'ItemID': 'ELEC007',
   'ItemName': 'Mi 10000mAh Power Bank',
   'ClosingStock': 22,
   'ReorderLevel': 25},
  {'ItemID': 'ELEC008',
   'ItemName': 'Anker USB-C Fast Charger 65W',
   'ClosingStock': -22,
   'ReorderLevel': 20},
  {'ItemID': 'ELEC009',
   'ItemName': 'Logitech M185 Wireless Mouse',
   'ClosingStock': 11,
   'ReorderLevel': 25},
  {'ItemID': 'ELEC010',
   'ItemName': 'TP-Link TL-WR841N

In [7]:
from analysis.monthly_growth import compute_monthly_growth
compute_monthly_growth(state.store)

2026-03-04 18:20:57 | INFO     | analysis.monthly_growth | Monthly growth computed.


{'monthly': [{'MonthName': 'Jan',
   'MonthRevenue': 3403184,
   'MonthProfit': 621973.85,
   'RevGrowth': nan,
   'ProfGrowth': nan},
  {'MonthName': 'Feb',
   'MonthRevenue': 2959937,
   'MonthProfit': 510366.99,
   'RevGrowth': -13.02,
   'ProfGrowth': -17.94},
  {'MonthName': 'Mar',
   'MonthRevenue': 4360364,
   'MonthProfit': 900512.94,
   'RevGrowth': 47.31,
   'ProfGrowth': 76.44},
  {'MonthName': 'Apr',
   'MonthRevenue': 4511046,
   'MonthProfit': 940440.38,
   'RevGrowth': 3.46,
   'ProfGrowth': 4.43},
  {'MonthName': 'May',
   'MonthRevenue': 5044218,
   'MonthProfit': 1074822.31,
   'RevGrowth': 11.82,
   'ProfGrowth': 14.29},
  {'MonthName': 'Jun',
   'MonthRevenue': 3851110,
   'MonthProfit': 752422.24,
   'RevGrowth': -23.65,
   'ProfGrowth': -30.0},
  {'MonthName': 'Jul',
   'MonthRevenue': 3427241,
   'MonthProfit': 621648.77,
   'RevGrowth': -11.01,
   'ProfGrowth': -17.38},
  {'MonthName': 'Aug',
   'MonthRevenue': 4893385,
   'MonthProfit': 1036926.84,
   'RevGrowt

In [8]:
from analysis.breakeven import compute_breakeven
compute_breakeven(state.store)

2026-03-04 18:20:57 | INFO     | analysis.breakeven | Break-even: BEU=1,382  MoS=93.46%


{'weighted_avg_cost_npr': 2031.32,
 'avg_contrib_per_unit_npr': 3158.58,
 'total_opex_npr': 4364012,
 'overall_breakeven_units': 1382.0,
 'actual_units_sold': 21141,
 'margin_of_safety_pct': 93.46,
 'top_20_easiest_breakeven': [{'ItemID': 'ELEC011',
   'ItemName': 'HP 15s Intel Core i3 Laptop',
   'BreakEvenUnits': 71.58,
   'ContribPerUnit': 60967.68},
  {'ItemID': 'ELEC002',
   'ItemName': 'Realme Narzo 70 Pro',
   'BreakEvenUnits': 136.51,
   'ContribPerUnit': 31967.68},
  {'ItemID': 'ELEC001',
   'ItemName': 'Samsung Galaxy A15 5G',
   'BreakEvenUnits': 168.06,
   'ContribPerUnit': 25967.68},
  {'ItemID': 'ELEC003',
   'ItemName': 'Xiaomi Redmi Note 13',
   'BreakEvenUnits': 194.24,
   'ContribPerUnit': 22467.68},
  {'ItemID': 'ELEC013',
   'ItemName': 'Lenovo Smart Tab M10 Plus',
   'BreakEvenUnits': 208.13,
   'ContribPerUnit': 20967.68},
  {'ItemID': 'HOME012',
   'ItemName': 'Livpure Smart Water Purifier',
   'BreakEvenUnits': 265.0,
   'ContribPerUnit': 16467.68},
  {'ItemID':

In [9]:
from analysis.cashflow import compute_cashflow
compute_cashflow(state.store)

2026-03-04 18:20:57 | INFO     | analysis.cashflow | Cashflow: NetCashMovement=30,260,772


{'total_cash_inflow_npr': 59510472,
 'total_cash_outflow_npr': 29249700,
 'net_cash_movement_npr': 30260772,
 'monthly_cashflow': [{'MonthNum': 1,
   'CashIn': 3403184,
   'CashOut': 4221005,
   'NetCash': -817821},
  {'MonthNum': 2, 'CashIn': 2959937, 'CashOut': 1740862, 'NetCash': 1219075},
  {'MonthNum': 3, 'CashIn': 4360364, 'CashOut': 3780484, 'NetCash': 579880},
  {'MonthNum': 4, 'CashIn': 4511046, 'CashOut': 3127204, 'NetCash': 1383842},
  {'MonthNum': 5, 'CashIn': 5044218, 'CashOut': 3149078, 'NetCash': 1895140},
  {'MonthNum': 6, 'CashIn': 3851110, 'CashOut': 2932315, 'NetCash': 918795},
  {'MonthNum': 7, 'CashIn': 3427241, 'CashOut': 433865, 'NetCash': 2993376},
  {'MonthNum': 8, 'CashIn': 4893385, 'CashOut': 1419130, 'NetCash': 3474255},
  {'MonthNum': 9, 'CashIn': 5539147, 'CashOut': 1187885, 'NetCash': 4351262},
  {'MonthNum': 10, 'CashIn': 6412480, 'CashOut': 1161905, 'NetCash': 5250575},
  {'MonthNum': 11, 'CashIn': 6576137, 'CashOut': 511345, 'NetCash': 6064792},
  {'Mo

## Generate reports

In [10]:
from reports.report_generator import generate_pdf, generate_excel
print(generate_pdf(state.store))
print(generate_excel(state.store))

2026-03-04 18:20:58 | INFO     | analysis.profitability | Profitability: NetProfit=12,202,347  GPM=27.84%
2026-03-04 18:20:58 | INFO     | analysis.discounts | Discounts: 2.85% of gross revenue
2026-03-04 18:20:58 | INFO     | analysis.inventory | Inventory: turnover=1.73x  DIO=211.5d
2026-03-04 18:20:58 | INFO     | analysis.products | Product metrics computed.
2026-03-04 18:20:58 | INFO     | analysis.expenses | Expenses: OpEx%=7.33
2026-03-04 18:20:58 | INFO     | analysis.monthly_growth | Monthly growth computed.
2026-03-04 18:20:58 | INFO     | analysis.breakeven | Break-even: BEU=1,382  MoS=93.46%
2026-03-04 18:20:58 | INFO     | analysis.cashflow | Cashflow: NetCashMovement=30,260,772
2026-03-04 18:20:58 | INFO     | reports.report_generator | PDF saved: data\exports\BI_Report.pdf
2026-03-04 18:20:58 | INFO     | analysis.profitability | Profitability: NetProfit=12,202,347  GPM=27.84%
2026-03-04 18:20:58 | INFO     | analysis.discounts | Discounts: 2.85% of gross revenue
2026-03

data\exports\BI_Report.pdf


2026-03-04 18:20:58 | INFO     | reports.report_generator | Excel saved: data\exports\BI_Report.xlsx


data\exports\BI_Report.xlsx
